# Proyecto II: Detección de anomalías industriales mediante destilado de modelos y autoencoders

**Curso:** Inteligencia Artificial, Escuela de Ingeniería en Computación, Instituto Tecnológico de Costa Rica

**Integrantes:**
- Sebastián Chacón - 2023221908
- Daniel Pulido - 2019196502
- Sebastián Muñoz - 2023090681

**Notebook 2 de 4: modelos A y B.** Entrenar el clasificador CNN con la estructura de ResNet-18 hasta `conv3_x`, desde cero (modelo A) y por destilado teacher student a partir de un ResNet-18 completo (modelo B), con al menos 3 combinaciones de hiperparámetros cada uno.

El proyecto se divide en cuatro notebooks que comparten la configuración de `conf/` y el código de `src/`:
1. `datos.ipynb`: exploración de MVTec AD y `LightningDataModule`.
2. `modelos_ab.ipynb`: modelo A (CNN desde cero) y modelo B (CNN destilada).
3. `modelo_c.ipynb`: modelo C (autoencoder U-Net).
4. `anomalias.ipynb`: detección de anomalías con embeddings y análisis con DBSCAN.

## Contenido
1. Configuración y entorno
2. Modelo A, CNN desde cero
3. Modelo B, CNN destilada
4. Comparación de los modelos A y B

## 1. Configuración y entorno

### 1.1 Imports y configuración con Hydra
Las clases (DataModule, modelos y utilidades de entrenamiento) están en `src/` como scripts auxiliares, y este notebook las importa y ejecuta. Para eso se agrega la raíz del repositorio a `sys.path`. Con `autoreload` los cambios en `src/` se cargan sin reiniciar el kernel.

El decorador `@hydra.main` está pensado para scripts que se ejecutan desde la terminal y no funciona dentro de Jupyter. Por eso `load_config` usa la API de composición de Hydra (`initialize_config_dir` y `compose`), que es la que recomienda su documentación para notebooks. Recibe una lista de overrides, por ejemplo `["+experiment=cnn_scratch/lr3e-4_bs32"]`, y así cada corrida varía sus hiperparámetros sin tocar los archivos YAML.

In [1]:
# Imports, repository root and Hydra configuration
%load_ext autoreload
%autoreload 2

import logging
import sys
import warnings
from pathlib import Path

# Hide noise that does not affect the results: torch's "triton not found" (no triton on Windows)
logging.getLogger("torch.utils.flop_counter").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# The notebook lives in notebooks/, the repository root is its parent folder
PROJECT_ROOT = Path.cwd().resolve().parent
assert (PROJECT_ROOT / "conf").is_dir(), "Run the notebook from the notebooks/ folder"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import lightning as L
import numpy as np
import pandas as pd
import torch
import wandb
from hydra.utils import instantiate
from omegaconf import OmegaConf

from src.utils import load_config, resolve_path

cfg = load_config()
print(OmegaConf.to_yaml(cfg))

data:
  _target_: src.data.MVTecDataModule
  _convert_: all
  root: data/mvtec_ad
  categories:
  - carpet
  - leather
  - tile
  - wood
  - bottle
  - cable
  - capsule
  - hazelnut
  - metal_nut
  - pill
  image_size: 224
  batch_size: 32
  num_workers: 0
  cache_dir: ${paths.output_dir}/cache
  val_split: 0.2
  seed: ${seed}
  mean:
  - 0.485
  - 0.456
  - 0.406
  std:
  - 0.229
  - 0.224
  - 0.225
model:
  _target_: src.models.cnn.CNNClassifierModule
  _convert_: all
  name: cnn_scratch
  num_classes: ${len:${data.categories}}
  z_dim: null
  head:
    hidden_dim: 256
    dropout: 0.3
  optimizer:
    _target_: torch.optim.AdamW
    _partial_: true
    lr: 0.001
    weight_decay: 0.0001
  scheduler: null
trainer:
  _target_: lightning.Trainer
  max_epochs: 50
  accelerator: auto
  devices: 1
  precision: 16-mixed
  log_every_n_steps: 10
  enable_progress_bar: false
callbacks:
  early_stopping:
    _target_: lightning.pytorch.callbacks.EarlyStopping
    monitor: val/loss
    mode: m

### 1.2 Semilla y GPU
Se fija la semilla de Python, NumPy y PyTorch para que los resultados sean reproducibles, y se verifica que la GPU esté disponible.

In [2]:
# Seed, GPU and library versions
L.seed_everything(cfg.seed)
# Lets float32 matrix multiplications use the GPU Tensor Cores (TF32): faster, negligible precision change
torch.set_float32_matmul_precision("high")

print("PyTorch", torch.__version__, "| Lightning", L.__version__, "| wandb", wandb.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}, {props.total_memory / 1024**3:.1f} GB de VRAM")

Seed set to 42


PyTorch 2.14.1+cu130 | Lightning 2.6.6 | wandb 0.31.2
CUDA disponible: True
GPU: NVIDIA GeForce RTX 4060, 8.0 GB de VRAM


### 1.3 Datos
Se construye el `MVTecDataModule` desde `conf/data/mvtec.yaml` (está descrito y verificado en `datos.ipynb`). Aquí se usa un batch de entrenamiento para comprobar las formas de cada etapa de la red.

In [3]:
# DataModule and one training batch to check the shapes of the network
datamodule = instantiate(cfg.data)
datamodule.prepare_data()
datamodule.setup()
categories = datamodule.categories
batch = next(iter(datamodule.train_dataloader()))
print(f"train {len(datamodule.train_set)} | val {len(datamodule.val_set)} | test {len(datamodule.test_set)}")
print("batch:", tuple(batch["image"].shape), "| categorias:", categories)

train 2025 | val 507 | test 1194
batch: (32, 3, 224, 224) | categorias: ['carpet', 'leather', 'tile', 'wood', 'bottle', 'cable', 'capsule', 'hazelnut', 'metal_nut', 'pill']


## 2. Modelo A, CNN desde cero

Código en `src/models/cnn.py`, configuración en `conf/model/cnn_scratch.yaml` y combinaciones en `conf/experiment/cnn_scratch/`.

### 2.1 Backbone: ResNet-18 hasta `conv3_x`
El enunciado pide seguir la estructura de ResNet-18 en sus tres primeras etapas (Figura 1 del enunciado):

| Etapa | Capas | Salida con entrada 224×224 |
|---|---|---|
| `conv1` | conv 7×7, 64 filtros, stride 2 + BatchNorm + ReLU | 64 × 112 × 112 |
| `conv2_x` | max pool 3×3 stride 2 + 2 bloques residuales [3×3, 64; 3×3, 64] | 64 × 56 × 56 |
| `conv3_x` | 2 bloques residuales [3×3, 128; 3×3, 128], el primero con stride 2 | 128 × 28 × 28 |

En lugar de reescribir esas capas se toman del `resnet18` de torchvision con `weights=None`. Así la estructura es idéntica a la del ResNet-18 que hará de teacher en el modelo B (en torchvision `conv2_x` es `maxpool + layer1` y `conv3_x` es `layer2`), y con `weights=None` los pesos empiezan aleatorios (inicialización de Kaiming), como pide el enunciado para el modelo A. La clase `ResNet18Stem` expone además `forward_features`, que devuelve la salida de cada etapa. El modelo A no la usa, pero el modelo B la necesita para comparar sus features con las del teacher.

In [4]:
# Output shape of each stage of the backbone
from src.models.cnn import CNNClassifier, CNNClassifierModule, ResNet18Stem

stem = ResNet18Stem()
with torch.no_grad():
    features = stem.forward_features(batch["image"][:4])
print(f"{'entrada':8s} {tuple(batch['image'][:4].shape)}")
for name, f in features.items():
    print(f"{name:8s} {tuple(f.shape)}")

entrada  (4, 3, 224, 224)
conv1    (4, 64, 112, 112)
conv2_x  (4, 64, 56, 56)
conv3_x  (4, 128, 28, 28)


### 2.2 Clasificador y `get_embedding`
Después de `conv3_x` se aplica *global average pooling*, que promedia cada uno de los 128 canales sobre las 28×28 posiciones. Ese vector de 128 valores es el **embedding** del modelo. Es la salida de las capas convolucionales que pide el enunciado y sobre él se construye el detector de anomalías. El método `get_embedding(x)` lo devuelve directamente.

El clasificador es una cabeza FC de diseño libre: `128 -> hidden_dim -> 10` con ReLU y dropout. La capa oculta le da a la cabeza capacidad para separar las categorías sin forzar al embedding a ser linealmente separable, y el dropout reduce el sobreajuste en un conjunto de unas 2000 imágenes. El embedding se toma **antes** del clasificador para que no dependa de la tarea de clasificación más de lo necesario.

La tarea de clasificar la categoría es fácil a propósito. Su función es obligar al backbone a aprender representaciones de las imágenes normales, y por eso el accuracy no es la métrica principal del proyecto. Lo que importa es qué tan útil es el embedding para separar lo normal de lo anómalo, y eso se mide en `anomalias.ipynb`.

La configuración tiene además `z_dim` (por defecto `null`). Con un número agrega una proyección lineal `128 -> z_dim` y el embedding se toma después de ella. Así la dimensión del embedding también se puede variar desde Hydra.

In [5]:
# Embedding and logits shapes, and number of parameters against a full ResNet-18
from torchvision.models import resnet18

from src.reporting import count_parameters

cfg_a = load_config(["+experiment=cnn_scratch/lr1e-3_bs32"])
net = CNNClassifier(num_classes=len(categories), z_dim=cfg_a.model.z_dim, **cfg_a.model.head)
with torch.no_grad():
    x = batch["image"][:4]
    print("embedding:", tuple(net.get_embedding(x).shape), "| logits:", tuple(net(x).shape))

resnet_params = count_parameters(resnet18(weights=None))
params = pd.Series({
    "backbone (conv1 a conv3_x)": count_parameters(net.backbone),
    "clasificador FC": count_parameters(net.head),
    "modelo A completo": count_parameters(net),
    "ResNet-18 completo (referencia)": resnet_params,
}, name="parametros").to_frame()
params["millones"] = (params["parametros"] / 1e6).round(3)
params["% de ResNet-18"] = (100 * params["parametros"] / resnet_params).round(1)
params

embedding: (4, 128) | logits: (4, 10)


,parametros,millones,% de ResNet-18
backbone (conv1 a conv3_x),683072,0.683,5.8
clasificador FC,35594,0.036,0.3
modelo A completo,718666,0.719,6.1
ResNet-18 completo (referencia),11689512,11.690,100.0


### 2.3 `LightningModule`
`CNNClassifierModule` envuelve la red y define el entrenamiento:

- **`training_step`, `validation_step` y `test_step`** llaman a un `shared_step` común, que calcula la entropía cruzada entre los logits y la categoría de la imagen y registra `<split>/loss` y `<split>/acc` por época. Las tres etapas usan exactamente el mismo cálculo y solo cambia el prefijo.
- **`configure_optimizers`** construye el optimizador y el scheduler opcional a partir de los objetos parciales que crea Hydra (`_partial_: true`), porque los parámetros del modelo solo existen en ese momento.
- **`save_hyperparameters(logger=False)`** guarda la configuración del modelo dentro del checkpoint. No se envía al logger porque `run_experiment` ya registra en W&B la configuración completa de Hydra.

El módulo se construye con `instantiate(cfg.model)`, que lee el `_target_` de `conf/model/cnn_scratch.yaml` y le pasa el resto de las llaves como argumentos.

In [6]:
# Model A built from its Hydra config
model_a = instantiate(cfg_a.model)
print(L.pytorch.utilities.model_summary.ModelSummary(model_a, max_depth=3))

   | Name                 | Type              | Params | Mode  | FLOPs
----------------------------------------------------------------------------
0  | net                  | CNNClassifier     | 718 K  | train | 0    
1  | net.backbone         | ResNet18Stem      | 683 K  | train | 0    
2  | net.backbone.conv1   | Sequential        | 9.5 K  | train | 0    
3  | net.backbone.conv2_x | Sequential        | 147 K  | train | 0    
4  | net.backbone.conv3_x | Sequential        | 525 K  | train | 0    
5  | net.pool             | AdaptiveAvgPool2d | 0      | train | 0    
6  | net.projection       | Identity          | 0      | train | 0    
7  | net.head             | Sequential        | 35.6 K | train | 0    
8  | net.head.0           | Linear            | 33.0 K | train | 0    
9  | net.head.1           | ReLU              | 0      | train | 0    
10 | net.head.2           | Dropout           | 0      | train | 0    
11 | net.head.3           | Linear            | 2.6 K  | train | 0    


### 2.4 Combinaciones de hiperparámetros
El enunciado pide entrenar cada modelo con al menos 3 combinaciones de hiperparámetros. Cada una es un archivo de `conf/experiment/cnn_scratch/` que parte de `conf/model/cnn_scratch.yaml` y cambia solo lo que la distingue:

- **`A-cnn_scratch-lr1e-3_bs32`:** AdamW con learning rate constante de 1e-3 y batch de 32. Es el punto de referencia.
- **`A-cnn_scratch-lr3e-4_bs32`:** igual que la anterior pero con learning rate de 3e-4. Pasos más pequeños y estables, para aislar el efecto del learning rate.
- **`A-cnn_scratch-lr1e-3cos_bs64`:** batch de 64 (gradientes menos ruidosos y la mitad de pasos por época) y learning rate que baja de 1e-3 a 1e-5 siguiendo un coseno durante 20 épocas, con pasos grandes al inicio y ajustes finos al final.

Todas usan `EarlyStopping` sobre `val/loss` con paciencia de 5 épocas y guardan el checkpoint con la menor `val/loss`. El nombre de cada corrida sigue el formato `<letra del modelo>-<modelo>-<hiperparámetros>`, y es el mismo en W&B, en la carpeta del checkpoint y en el archivo de resultados. En W&B las corridas se agrupan por modelo (`group`), llevan la descripción como nota y tienen un tag por hiperparámetro (`lr=3e-4`, `bs=32`, `lr_coseno`) para filtrarlas. La tabla muestra los valores efectivos de cada combinación, tal como los compone Hydra.

In [7]:
# Effective hyperparameters of each combination (composed with Hydra)
EXPERIMENTS_A = ["cnn_scratch/lr1e-3_bs32", "cnn_scratch/lr3e-4_bs32", "cnn_scratch/lr1e-3cos_bs64"]


def hyperparameters_table(experiments):
    rows = []
    for experiment in experiments:
        c = load_config([f"+experiment={experiment}"])
        scheduler = c.model.get("scheduler")
        lr = c.model.optimizer.lr
        rows.append({
            "corrida": c.name,
            "batch": c.data.batch_size,
            "lr": f"{lr} -> {scheduler.eta_min} (coseno)" if scheduler else f"{lr} (constante)",
            "weight decay": c.model.optimizer.weight_decay,
            "z_dim": c.model.z_dim or "128 (sin proyeccion)",
            "capa oculta": c.model.head.hidden_dim,
            "dropout": c.model.head.dropout,
            "max epocas": c.trainer.max_epochs,
            "paciencia": c.callbacks.early_stopping.patience,
            "tags W&B": ", ".join(c.tags),
            "descripcion": c.description,
        })
    return pd.DataFrame(rows).set_index("corrida")


hyperparameters_table(EXPERIMENTS_A)

,batch,lr,weight decay,z_dim,capa oculta,dropout,max epocas,paciencia,tags W&B,descripcion
corrida,,,,,,,,,,
A-cnn_scratch-lr1e-3_bs32,32,0.001 (constante),0.0001,128 (sin proyeccion),256,0.3,50,5,"A, cnn_scratch, lr=1e-3, bs=32, lr_constante","Modelo A, AdamW lr 1e-3 constante, batch 32"
A-cnn_scratch-lr3e-4_bs32,32,0.0003 (constante),0.0001,128 (sin proyeccion),256,0.3,50,5,"A, cnn_scratch, lr=3e-4, bs=32, lr_constante","Modelo A, AdamW lr 3e-4 constante, batch 32"
A-cnn_scratch-lr1e-3cos_bs64,64,0.001 -> 1e-05 (coseno),0.0001,128 (sin proyeccion),256,0.3,20,5,"A, cnn_scratch, lr=1e-3, bs=64, lr_coseno","Modelo A, AdamW lr coseno 1e-3 -> 1e-5 en 20 e..."


### 2.5 Prueba rápida
Antes de entrenar se ejecuta un `fast_dev_run`: un batch de entrenamiento y uno de validación, sin logger ni checkpoints. Sirve para detectar errores de forma o de configuración en segundos, antes de crear corridas en W&B.

In [8]:
# fast_dev_run: one training and one validation batch, without logger or checkpoints
from src.training import build_callbacks, build_trainer, run_experiment

smoke_cfg = load_config(["+experiment=cnn_scratch/lr1e-3_bs32", "+trainer.fast_dev_run=true"])
smoke_callbacks = {k: v for k, v in build_callbacks(smoke_cfg).items() if k != "lr_monitor"}
build_trainer(smoke_cfg, smoke_callbacks).fit(instantiate(smoke_cfg.model), datamodule=datamodule)
print("fast_dev_run OK")

Using 16bit Automatic Mixed Precision (AMP)


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Running in `fast_dev_run` mode: will run the requested loop using 1 batch(es). Logging and checkpointing is suppressed.


C:\Users\Chaco\OneDrive\Documents\GitHub\MVTecDestilado\.venv\Lib\site-packages\lightning\pytorch\callbacks\model_checkpoint.py:881: Checkpoint directory C:\Users\Chaco\OneDrive\Documents\GitHub\MVTecDestilado\outputs\checkpoints\A-cnn_scratch-lr1e-3_bs32 exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


C:\Users\Chaco\OneDrive\Documents\GitHub\MVTecDestilado\.venv\Lib\site-packages\lightning\pytorch\utilities\model_summary\model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name | Type          | Params | Mode  | FLOPs
-------------------------------------------------------
0 | net  | CNNClassifier | 718 K  | train | 0    
-------------------------------------------------------
718 K     Trainable params
0         Non-trainable params
718 K     Total params
2.875     Total estimated model params size (MB)
44        Modules in train mode
0         Modules in eval mode
0         Total Flops


C:\Users\Chaco\OneDrive\Documents\GitHub\MVTecDestilado\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
C:\Users\Chaco\OneDrive\Documents\GitHub\MVTecDestilado\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


epoca   1 | train/acc 0.1250 | train/loss 2.3191 | val/acc 1.0000 | val/loss 1.8779


`Trainer.fit` stopped: `max_steps=1` reached.


fast_dev_run OK


### 2.6 Entrenamiento
`run_experiment` (en `src/training.py`) entrena una combinación completa. Construye el DataModule, el modelo, los callbacks, el logger de W&B y el Trainer desde la configuración, ejecuta `fit`, evalúa el mejor checkpoint con `test` y guarda un resumen en `outputs/results/<corrida>.json`. Si una corrida ya se hizo, es decir si existen su resumen y su checkpoint, se omite su entrenamiento y se carga el resumen guardado. Así el notebook se puede volver a ejecutar completo sin repetir entrenamientos ni crear corridas duplicadas en W&B. Para repetir el entrenamiento se pone `RETRAIN = True`.

In [9]:
# Train model A with every combination (runs that already have a results file are loaded, not retrained)
RETRAIN = False
results_a = {}
for experiment in EXPERIMENTS_A:
    cfg_run = load_config([f"+experiment={experiment}"])
    results_a[cfg_run.name] = run_experiment(cfg_run, retrain=RETRAIN)

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\Chaco\_netrc.
wandb: Currently logged in as: chaco17 to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


>> A-cnn_scratch-lr1e-3_bs32 | Modelo A, AdamW lr 1e-3 constante, batch 32 | https://forge.coreweave.com/wandb/danielpulido-instituto-tecnol-gico-de-costa-rica/MVTecDestilado/runs/1kd56j2p


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\Chaco\OneDrive\Documents\GitHub\MVTecDestilado\.venv\Lib\site-packages\lightning\pytorch\utilities\model_summary\model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name | Type          | Params | Mode  | FLOPs
-------------------------------------------------------
0 | net  | CNNClassifier | 718 K  | train | 0    
-------------------------------------------------------
718 K     Trainable params
0         Non-trainable params
718 K     Total params
2.875     Total estimated model params size (MB)
44        Modules in train mode
0         Modules in eval mode
0         Total Flops
c:\Users\Chaco\OneDrive\Documents\GitHub\MVTecDestilado\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `nu

epoca   1 | train/acc 0.9249 | train/loss 0.3251 | val/acc 1.0000 | val/loss 0.0088
epoca   2 | train/acc 0.9793 | train/loss 0.0814 | val/acc 1.0000 | val/loss 0.0016
epoca   3 | train/acc 0.9956 | train/loss 0.0188 | val/acc 1.0000 | val/loss 0.0070
epoca   4 | train/acc 0.9936 | train/loss 0.0242 | val/acc 1.0000 | val/loss 0.0122
epoca   5 | train/acc 0.9980 | train/loss 0.0085 | val/acc 1.0000 | val/loss 0.0002
epoca   6 | train/acc 0.9857 | train/loss 0.0710 | val/acc 1.0000 | val/loss 0.0004
epoca   7 | train/acc 0.9916 | train/loss 0.0278 | val/acc 1.0000 | val/loss 0.0003
epoca   8 | train/acc 0.9980 | train/loss 0.0076 | val/acc 1.0000 | val/loss 0.0001
epoca   9 | train/acc 0.9975 | train/loss 0.0095 | val/acc 0.9566 | val/loss 0.0805
epoca  10 | train/acc 0.9911 | train/loss 0.0479 | val/acc 1.0000 | val/loss 0.0104
epoca  11 | train/acc 0.9985 | train/loss 0.0077 | val/acc 1.0000 | val/loss 0.0006
epoca  12 | train/acc 0.9802 | train/loss 0.0871 | val/acc 1.0000 | val/loss

Restoring states from the checkpoint path at C:\Users\Chaco\OneDrive\Documents\GitHub\MVTecDestilado\outputs\checkpoints\A-cnn_scratch-lr1e-3_bs32\best.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at C:\Users\Chaco\OneDrive\Documents\GitHub\MVTecDestilado\outputs\checkpoints\A-cnn_scratch-lr1e-3_bs32\best.ckpt
c:\Users\Chaco\OneDrive\Documents\GitHub\MVTecDestilado\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


... (12 outputs left out)

epoca   1 | train/acc 0.8993 | train/loss 0.4862 | val/acc 1.0000 | val/loss 0.0089
epoca   2 | train/acc 1.0000 | train/loss 0.0109 | val/acc 0.9961 | val/loss 0.0101
epoca   3 | train/acc 0.9995 | train/loss 0.0076 | val/acc 1.0000 | val/loss 0.0005
epoca   4 | train/acc 0.9985 | train/loss 0.0045 | val/acc 0.8422 | val/loss 0.7408
epoca   5 | train/acc 0.9896 | train/loss 0.0281 | val/acc 0.8679 | val/loss 0.5620
epoca   6 | train/acc 0.9837 | train/loss 0.0695 | val/acc 0.9941 | val/loss 0.0263
epoca   7 | train/acc 0.9995 | train/loss 0.0054 | val/acc 1.0000 | val/loss 0.0006
epoca   8 | train/acc 1.0000 | train/loss 0.0016 | val/acc 1.0000 | val/loss 0.0001
epoca   9 | train/acc 1.0000 | train/loss 0.0006 | val/acc 1.0000 | val/loss 0.0001
epoca  10 | train/acc 1.0000 | train/loss 0.0006 | val/acc 1.0000 | val/loss 0.0001
epoca  11 | train/acc 1.0000 | train/loss 0.0007 | val/acc 1.0000 | val/loss 0.0001
epoca  12 | train/acc 1.0000 | train/loss 0.0006 | val/acc 1.0000 | val/loss

`Trainer.fit` stopped: `max_epochs=20` reached.
Restoring states from the checkpoint path at C:\Users\Chaco\OneDrive\Documents\GitHub\MVTecDestilado\outputs\checkpoints\A-cnn_scratch-lr1e-3cos_bs64\best.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at C:\Users\Chaco\OneDrive\Documents\GitHub\MVTecDestilado\outputs\checkpoints\A-cnn_scratch-lr1e-3cos_bs64\best.ckpt
c:\Users\Chaco\OneDrive\Documents\GitHub\MVTecDestilado\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇██
lr-AdamW,████▇▇▇▆▆▅▄▄▃▃▂▂▂▁▁▁
test/acc,▁
test/loss,▁
train/acc,▁███▇▇██████████████
train/loss,█▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
trainer/global_step,▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇███
val/acc,███▁▂███████████████
val/loss,▁▁▁█▆▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,20
lr-AdamW,2e-05


OK A-cnn_scratch-lr1e-3cos_bs64: 20 epocas en 0.8 min | mejor epoca 20 (val/loss 0.0000) | test {'test/loss': 7.317723066080362e-05, 'test/acc': 1.0}


### 2.7 Resultados y análisis de overfitting
La tabla resume cada corrida en su mejor época (la del checkpoint). El primer gráfico tiene una columna por corrida, con la pérdida (en escala logarítmica, porque va de 1 a 1e-5) y el accuracy de entrenamiento y validación por época. La línea punteada vertical marca la mejor época. El segundo gráfico compara la pérdida de validación de las tres corridas. Si la pérdida de validación sube mientras la de entrenamiento sigue bajando, el modelo está sobreajustando, y `EarlyStopping` debería detener el entrenamiento poco después.

Los gráficos se hacen con plotly. Quedan guardados dentro del notebook, así que se ven sin una cuenta de W&B, y son interactivos (zoom y valores al pasar el mouse). La misma figura se registra en una run de análisis de W&B, cuyo link queda impreso. Esa run tiene un id fijo, así que volver a ejecutar la celda la actualiza en lugar de crear otra. Las curvas de A se registran en la run `analisis-modelo-A`.

In [10]:
# Summary of the model A runs
from src.reporting import runs_table

with pd.option_context("display.float_format", "{:.4g}".format):
    display(runs_table(results_a))

,epocas,parada temprana,mejor epoca,train/loss,val/loss,test/loss,train/acc,val/acc,test/acc,minutos
corrida,,,,,,,,,,
A-cnn_scratch-lr1e-3_bs32,13,True,8,0.007642,5.964e-05,0.0001135,0.998,1,1,0.52
A-cnn_scratch-lr3e-4_bs32,12,True,7,0.003141,0.000281,0.0003416,1,1,1,0.51
A-cnn_scratch-lr1e-3cos_bs64,20,False,20,0.0003411,4.489e-05,7.318e-05,1,1,1,0.84


In [11]:
# Training and validation curves of model A (plotly), also logged to W&B
from src.reporting import bar_figure, log_figures, runs_figure, train_val_figure

fig_a_curves = train_val_figure(results_a, [("loss", "perdida (entropia cruzada)", True), ("acc", "accuracy", False)],
                                title="Modelo A: entrenamiento y validacion por epoca")
fig_a_val = runs_figure(results_a, "val/loss", "perdida de validacion", log_y=True,
                        title="Modelo A: perdida de validacion de las tres corridas")
fig_a_curves.show()
fig_a_val.show()
log_figures(cfg, "analisis-modelo-a", "analisis-modelo-A", "Curvas de entrenamiento y validacion del modelo A",
            {"curvas/train_val": fig_a_curves, "curvas/val_loss": fig_a_val})

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\Chaco\_netrc.


wandb: Currently logged in as: chaco17 to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


wandb: setting up run analisis-modelo-a


wandb: Tracking run with wandb version 0.31.2


wandb: Run data is saved locally in C:\Users\Chaco\OneDrive\Documents\GitHub\MVTecDestilado\outputs\wandb\run-20261127_121527-analisis-modelo-a
wandb: Run `wandb offline` to turn off syncing.


wandb: Resuming run analisis-modelo-A


... (4 outputs left out)

wandb: uploading wandb-metadata.json; uploading requirements.txt; uploading wandb-summary.json


wandb:  View run analisis-modelo-A at: https://forge.coreweave.com/wandb/danielpulido-instituto-tecnol-gico-de-costa-rica/MVTecDestilado/runs/analisis-modelo-a
wandb:  View project at: https://forge.coreweave.com/wandb/danielpulido-instituto-tecnol-gico-de-costa-rica/MVTecDestilado
wandb: Synced 4 W&B file(s), 2 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: C:\Users\Chaco\OneDrive\Documents\GitHub\MVTecDestilado\outputs\wandb\run-20261127_121527-analisis-modelo-a\logs


Graficos registrados en W&B: https://forge.coreweave.com/wandb/danielpulido-instituto-tecnol-gico-de-costa-rica/MVTecDestilado/runs/analisis-modelo-a


**Análisis.**

La clasificación de las 10 categorías se satura desde la primera época. Las tres corridas llegan a un `val/acc` de 1.000 en la época 1 y terminan con un `test/acc` de 1.000, porque las categorías se distinguen fácilmente por su color y su forma general. Por la misma razón, la pérdida de validación tampoco separa las combinaciones. Sus mejores valores son 6.0e-5 (`lr1e-3_bs32`, época 8), 2.8e-4 (`lr3e-4_bs32`, época 7) y 4.5e-5 (`lr1e-3cos_bs64`, época 20), y con pérdidas de ese orden el modelo ya asigna más del 99.99 % de probabilidad a la categoría correcta. Las diferencias entre corridas son ruido y no indican que una combinación sea mejor que otra.

Donde sí se ven diferencias es en la estabilidad. Con learning rate constante la `val/loss` de `lr1e-3_bs32` salta de 6.0e-5 a 8.1e-2 en la época 9 (con `val/acc` de 0.957) y la de `lr3e-4_bs32` sube a 0.39 en la época 11 (con `val/acc` de 0.911). La causa probable es que Adam da pasos de tamaño parecido al learning rate aunque el gradiente sea muy pequeño, así que cerca del mínimo sigue oscilando. El coseno de `lr1e-3cos_bs64` corrige este problema. También tiene picos mientras el learning rate es alto (`val/loss` de 0.74 en la época 4), pero desde la época 7 la curva baja de forma monótona hasta 4.5e-5 y es la única corrida que completa sus 20 épocas sin activar `EarlyStopping`.

No hay sobreajuste. Las pérdidas de entrenamiento y de validación terminan en el mismo orden, por debajo de 1e-2 en las tres corridas, y las paradas tempranas de `lr1e-3_bs32` (época 13) y `lr3e-4_bs32` (época 12) se deben a las oscilaciones y no a que la validación empeore de forma sostenida. Con `min_delta` de 0 y pérdidas tan pequeñas, cualquier oscilación cuenta como empeoramiento. Como la tarea se resuelve con muy poca información de la imagen, no hay razones para esperar que el backbone aprenda los detalles finos que distinguen un defecto, y por eso el modelo A queda como línea base. La calidad de su embedding se mide en `anomalias.ipynb`, donde se compara con la misma red sin entrenar, con el modelo B y con el ResNet-18 preentrenado.

## 3. Modelo B, CNN destilada

Código en `src/models/distill.py`, configuración en `conf/model/cnn_distill.yaml` y combinaciones en `conf/experiment/cnn_distill/`.

El modelo B usa exactamente la misma red que el modelo A (las tres primeras etapas de ResNet-18 con pesos aleatorios y el mismo clasificador FC). La diferencia está en cómo se entrena. Además de clasificar la categoría, imita lo que un ResNet-18 preentrenado en ImageNet (el *teacher*) calcula en sus capas intermedias. Así, cualquier diferencia entre A y B se puede atribuir al destilado.

### 3.1 Método de destilado: STFPM

**Qué es el destilado.** El destilado de conocimiento (*knowledge distillation*) entrena un modelo pequeño, el *student*, usando como guía un modelo grande ya entrenado, el *teacher*. El student no aprende solo de las etiquetas, también imita algo que produce el teacher. Según el conocimiento que se transfiere, los métodos se agrupan en tres familias (Gou et al., 2021):

- **De respuesta:** el student imita la salida final del teacher, como las probabilidades suavizadas con temperatura de Hinton et al. (2015).
- **De features:** el student imita los mapas de activación de capas intermedias, como en FitNets (Romero et al., 2015).
- **De relaciones:** el student imita cómo se relacionan las muestras entre sí, como las distancias entre pares de RKD (Park et al., 2019).

**Método elegido.** Se usa **STFPM** (*Student-Teacher Feature Pyramid Matching*, Wang et al., 2021), un método de destilado de features diseñado para detección de anomalías y evaluado en MVTec AD. Su idea es que el teacher, preentrenado en ImageNet, ya sabe describir bordes, texturas y formas. Si el student aprende a reproducir esas descripciones en imágenes normales, hereda ese conocimiento sin haber visto ImageNet.

En STFPM el teacher y el student tienen la misma arquitectura ResNet-18. El teacher está preentrenado y congelado, y el student empieza con pesos aleatorios. Para cada etapa $l$ elegida y cada posición $(h, w)$ del mapa de features, los vectores de canales del teacher $F_t^l(h,w)$ y del student $F_s^l(h,w)$ se normalizan con su norma L2 y se comparan con la mitad de su distancia euclidiana al cuadrado:

$$\ell^l = \frac{1}{H_l W_l} \sum_{h,w} \frac{1}{2} \left\| \frac{F_t^l(h,w)}{\|F_t^l(h,w)\|_2} - \frac{F_s^l(h,w)}{\|F_s^l(h,w)\|_2} \right\|_2^2 , \qquad \mathcal{L}_{\text{STFPM}} = \sum_l \ell^l$$

Como los dos vectores tienen norma 1, la expresión es igual a $1 - \cos(F_t^l(h,w), F_s^l(h,w))$. La pérdida exige que el patrón de activación de cada posición apunte en la misma dirección que el del teacher, sin importar su escala. Vale 0 cuando coinciden y crece hasta 2 cuando son opuestos. Se promedia sobre las posiciones y las imágenes del batch.

**Por qué STFPM.** De los métodos de destilado usados en detección de anomalías (STFPM, MKD, Reverse Distillation y EfficientAD), STFPM es el único que encaja sin cambios con lo que pide el enunciado:

- Su teacher es un ResNet-18 y su student tiene la misma arquitectura, igual que aquí.
- Nuestro student comparte con el teacher las etapas `conv2_x` y `conv3_x` con la misma forma (64×56×56 y 128×28×28), así que las features se comparan directamente, sin capas adaptadoras.
- La pérdida actúa sobre `conv3_x`, que es justamente la capa de la que sale el embedding.
- Se puede sumar a la entropía cruzada, así que el modelo B sigue siendo un clasificador.

Reverse Distillation y EfficientAD exigen que el student sea otra red (un decodificador o una red PDN), y el destilado de respuesta necesitaría reentrenar el teacher en las 10 categorías, una tarea que el modelo A ya resolvió al 100 % y que por eso transmite poca información.

**Adaptación a este proyecto.**

| | STFPM original | Modelo B |
|---|---|---|
| Teacher | ResNet-18 de ImageNet, congelado | Igual |
| Student | ResNet-18 completo, pesos aleatorios | Etapas `conv1` a `conv3_x` de ResNet-18 y un clasificador FC, pesos aleatorios |
| Etapas comparadas | `conv2_x`, `conv3_x` y `conv4_x` | `conv2_x` y `conv3_x`, las que tiene el student |
| Pérdida | $\mathcal{L}_{\text{STFPM}}$ | $\mathcal{L} = \text{CE} + \lambda \, \mathcal{L}_{\text{STFPM}}$ con $\lambda = 1$ |
| Datos de entrenamiento | Solo imágenes normales | Igual |
| Optimizador | SGD con learning rate 0.4 | AdamW con learning rate 3e-4 y coseno (y SGD con learning rate 0.05 como una de las combinaciones) |
| Resolución | 256×256 | 224×224, la del resto del proyecto |

La entropía cruzada se mantiene porque el enunciado define el modelo B como una variante del clasificador del modelo A. La resolución se mantiene en 224 para que los tres modelos usen exactamente los mismos datos.

**Detalle de implementación.** El teacher debe quedarse en modo `eval` durante todo el entrenamiento. Lightning llama `model.train()` al empezar cada época, y si el teacher pasara a modo entrenamiento sus capas BatchNorm actualizarían las estadísticas aprendidas en ImageNet con nuestras imágenes y dejaría de ser el mismo teacher. Por eso `DistillModule` redefine `train()` para devolver siempre el teacher a `eval`, sus parámetros tienen `requires_grad=False`, se ejecuta dentro de `torch.no_grad()` y el optimizador solo recibe los parámetros del student.

**Referencias.**
- Wang, G., Han, S., Ding, E. y Huang, D. (2021). *Student-Teacher Feature Pyramid Matching for Anomaly Detection*. BMVC. arXiv:2103.04257.
- Hinton, G., Vinyals, O. y Dean, J. (2015). *Distilling the Knowledge in a Neural Network*. arXiv:1503.02531.
- Romero, A. et al. (2015). *FitNets: Hints for Thin Deep Nets*. ICLR. arXiv:1412.6550.
- Gou, J., Yu, B., Maybank, S. J. y Tao, D. (2021). *Knowledge Distillation: A Survey*. IJCV. arXiv:2006.05525.

### 3.2 Teacher: ResNet-18 preentrenado
El teacher es el `resnet18` de torchvision con los pesos `IMAGENET1K_V1`. De él solo se ejecutan las etapas que se comparan con el student (`conv1`, `conv2_x` y `conv3_x`), porque `conv4_x`, `conv5_x` y su clasificador no intervienen en la pérdida. La celda comprueba que las features del teacher y del student tienen la misma forma en las etapas comparadas, que el teacher está congelado y cuántos parámetros tiene cada red.

In [12]:
# Teacher and student of model B: shapes of the compared stages, frozen teacher and parameter counts
from src.models.distill import DistillModule, stfpm_loss

cfg_b = load_config(["+experiment=cnn_distill/stfpm_l23"])
model_b = instantiate(cfg_b.model)

with torch.no_grad():
    x = batch["image"][:4]
    student_features = model_b.net.backbone.forward_features(x)
    teacher_features = model_b.teacher.forward_features(x)
for name in DistillModule.COMPARABLE_LAYERS:
    print(f"{name:8s} student {tuple(student_features[name].shape)} | teacher {tuple(teacher_features[name].shape)}")
print("teacher congelado:", not any(p.requires_grad for p in model_b.teacher.parameters()),
      "| teacher en eval:", not model_b.teacher.training)

params_b = pd.Series({
    "student (modelo B, igual al modelo A)": count_parameters(model_b.net),
    "teacher, etapas usadas (conv1 a conv3_x)": count_parameters(model_b.teacher),
    "teacher, ResNet-18 completo": resnet_params,
}, name="parametros").to_frame()
params_b["millones"] = (params_b["parametros"] / 1e6).round(3)
params_b["% de ResNet-18"] = (100 * params_b["parametros"] / resnet_params).round(1)
params_b

conv2_x  student (4, 64, 56, 56) | teacher (4, 64, 56, 56)
conv3_x  student (4, 128, 28, 28) | teacher (4, 128, 28, 28)
teacher congelado: True | teacher en eval: True


,parametros,millones,% de ResNet-18
"student (modelo B, igual al modelo A)",718666,0.719,6.1
"teacher, etapas usadas (conv1 a conv3_x)",683072,0.683,5.8
"teacher, ResNet-18 completo",11689512,11.690,100.0


### 3.3 Pérdida STFPM
`stfpm_loss` implementa la ecuación de la sección 3.1 para una etapa. Antes de entrenar se comprueba que se comporta como se espera:

- Entre el teacher y él mismo la pérdida debe ser 0.
- Entre el teacher y un student con pesos aleatorios debe ser alta, porque sus features no se parecen.
- Debe coincidir con $1 - \cos$ calculado con `torch.nn.functional.cosine_similarity`.

In [13]:
# Sanity checks of the STFPM loss
import torch.nn.functional as F

checks = {}
for name in DistillModule.COMPARABLE_LAYERS:
    checks[name] = {
        "teacher vs teacher": stfpm_loss(teacher_features[name], teacher_features[name]).item(),
        "teacher vs student aleatorio": stfpm_loss(student_features[name], teacher_features[name]).item(),
        "1 - coseno (referencia)": (1 - F.cosine_similarity(student_features[name], teacher_features[name], dim=1)).mean().item(),
    }
pd.DataFrame(checks).round(4)

,conv2_x,conv3_x
teacher vs teacher,0.0000,0.0000
teacher vs student aleatorio,0.4571,0.6474
1 - coseno (referencia),0.4571,0.6474


### 3.4 `LightningModule`
`DistillModule` sigue la misma estructura que el módulo del modelo A:

- **`shared_step`** calcula, para cualquier etapa, las features del student y del teacher, la entropía cruzada de la clasificación y la pérdida STFPM de cada etapa. La pérdida total es $\text{CE} + \lambda \sum_l \ell^l$ sobre las etapas elegidas en `distill.layers`.
- Registra `loss`, `ce`, `distill` y `acc`, y además la pérdida STFPM de **cada** etapa (`distill_conv2_x` y `distill_conv3_x`), aunque esa etapa no se use para entrenar. Así las combinaciones que destilan etapas distintas se pueden comparar con la misma métrica. `val/loss` no sirve para eso, porque suma una cantidad distinta de términos en cada combinación.
- **`configure_optimizers`** solo recibe los parámetros del student.
- **`train`** devuelve el teacher a `eval` cada vez que Lightning pone el módulo en modo entrenamiento.

El resumen muestra que los parámetros entrenables son los mismos 718 666 del modelo A. Los del teacher aparecen como no entrenables.

In [14]:
# Model B built from its Hydra config
print(L.pytorch.utilities.model_summary.ModelSummary(model_b, max_depth=2))

  | Name            | Type              | Params | Mode  | FLOPs
----------------------------------------------------------------------
0 | net             | CNNClassifier     | 718 K  | train | 0    
1 | net.backbone    | ResNet18Stem      | 683 K  | train | 0    
2 | net.pool        | AdaptiveAvgPool2d | 0      | train | 0    
3 | net.projection  | Identity          | 0      | train | 0    
4 | net.head        | Sequential        | 35.6 K | train | 0    
5 | teacher         | ResNet18Stem      | 683 K  | eval  | 0    
6 | teacher.conv1   | Sequential        | 9.5 K  | eval  | 0    
7 | teacher.conv2_x | Sequential        | 147 K  | eval  | 0    
8 | teacher.conv3_x | Sequential        | 525 K  | eval  | 0    
----------------------------------------------------------------------
718 K     Trainable params
683 K     Non-trainable params
1.4 M     Total params
5.607     Total estimated model params size (MB)
44        Modules in train mode
36        Modules in eval mode
0         Total

### 3.5 Combinaciones de hiperparámetros
Las tres combinaciones usan STFPM y cambian un solo aspecto con respecto a la referencia, para que su efecto se pueda atribuir:

- **`B-cnn_distill-stfpm_l23_lr3e-4`:** STFPM en `conv2_x` y `conv3_x`, AdamW con learning rate de 3e-4 que baja a 1e-5 con un coseno, batch de 32. Es la referencia. Usa el coseno porque en el modelo A fue lo que eliminó las oscilaciones del learning rate constante.
- **`B-cnn_distill-stfpm_l3_lr3e-4`:** igual, pero STFPM solo en `conv3_x`, la etapa de la que sale el embedding. Muestra si supervisar también la etapa intermedia ayuda.
- **`B-cnn_distill-stfpm_l23_sgd5e-2`:** igual que la referencia pero con SGD, el optimizador del artículo de STFPM, con momentum de 0.9 y weight decay de 1e-3 (los de la implementación de anomalib, porque el artículo no los indica). Muestra el efecto del optimizador.

Todas entrenan hasta 50 épocas con `EarlyStopping` sobre `val/loss` y guardan el checkpoint con la menor `val/loss`.

**Ajustes tras una primera ronda de entrenamientos.** Una primera versión de estas combinaciones mostró dos problemas, y los valores de arriba los corrigen:

- **Learning rate de 1e-3 con AdamW.** Las corridas con STFPM en dos etapas y en una etapa tuvieron un pico en la época 22, cuando el learning rate del coseno todavía era de unos 6e-4. La entropía cruzada de validación subió de 0.0004 a 0.51 y a 1.41, y `EarlyStopping` (con paciencia de 5) detuvo ambas en la época 26 aunque la pérdida STFPM seguía bajando unas 0.005 por época. Por eso el learning rate de AdamW baja a 3e-4 y la paciencia sube a 10 épocas.
- **SGD con learning rate de 0.4, el valor del artículo.** El clasificador colapsó. La entropía cruzada quedó fija en 2.31, que es $\ln 10$, el valor que se obtiene al predecir la misma probabilidad para las 10 categorías, y el accuracy de validación bajó a 0.15. El artículo usa ese valor porque entrena sin clasificador, solo con la pérdida STFPM. Con la cabeza de clasificación se necesita un learning rate menor, y 0.05 entrena de forma estable. A diferencia del modelo A, aquí `val/loss` no se satura, porque la parte de destilado mide qué tan bien imita el student al teacher en imágenes normales que no vio durante el entrenamiento. Para comparar las tres combinaciones entre sí se usa `val/distill_conv3_x`, que todas registran.

In [15]:
# Effective hyperparameters of each model B combination (composed with Hydra)
EXPERIMENTS_B = ["cnn_distill/stfpm_l23", "cnn_distill/stfpm_l3", "cnn_distill/stfpm_l23_sgd"]

rows = []
for experiment in EXPERIMENTS_B:
    c = load_config([f"+experiment={experiment}"])
    opt = c.model.optimizer
    rows.append({
        "corrida": c.name,
        "etapas STFPM": ", ".join(c.model.distill.layers),
        "peso STFPM": c.model.distill.weight,
        "optimizador": opt._target_.split(".")[-1],
        "lr": f"{opt.lr} -> {c.model.scheduler.eta_min} (coseno)",
        "momentum": opt.get("momentum", "-"),
        "weight decay": opt.weight_decay,
        "batch": c.data.batch_size,
        "max epocas": c.trainer.max_epochs,
        "paciencia": c.callbacks.early_stopping.patience,
        "teacher": c.model.teacher.weights,
        "tags W&B": ", ".join(c.tags),
    })
pd.DataFrame(rows).set_index("corrida")

,etapas STFPM,peso STFPM,optimizador,lr,momentum,weight decay,batch,max epocas,paciencia,teacher,tags W&B
corrida,,,,,,,,,,,
B-cnn_distill-stfpm_l23_lr3e-4,"conv2_x, conv3_x",1.0,AdamW,0.0003 -> 1e-05 (coseno),-,0.0001,32,50,10,IMAGENET1K_V1,"B, cnn_distill, stfpm, capas=conv2_x+conv3_x, ..."
B-cnn_distill-stfpm_l3_lr3e-4,conv3_x,1.0,AdamW,0.0003 -> 1e-05 (coseno),-,0.0001,32,50,10,IMAGENET1K_V1,"B, cnn_distill, stfpm, capas=conv3_x, adamw, l..."
B-cnn_distill-stfpm_l23_sgd5e-2,"conv2_x, conv3_x",1.0,SGD,0.05 -> 1e-05 (coseno),0.9,0.0010,32,50,10,IMAGENET1K_V1,"B, cnn_distill, stfpm, capas=conv2_x+conv3_x, ..."


### 3.6 Prueba rápida
Igual que con el modelo A, un `fast_dev_run` con un batch de entrenamiento y uno de validación comprueba que el modelo, la pérdida y el teacher funcionan antes de crear corridas en W&B.

In [16]:
# fast_dev_run of model B: one training and one validation batch, without logger or checkpoints
smoke_cfg_b = load_config(["+experiment=cnn_distill/stfpm_l23", "+trainer.fast_dev_run=true"])
smoke_callbacks_b = {k: v for k, v in build_callbacks(smoke_cfg_b).items() if k != "lr_monitor"}
smoke_model_b = instantiate(smoke_cfg_b.model)
build_trainer(smoke_cfg_b, smoke_callbacks_b).fit(smoke_model_b, datamodule=datamodule)
print("fast_dev_run OK | teacher sigue en eval:", not smoke_model_b.teacher.training)

Using 16bit Automatic Mixed Precision (AMP)


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Running in `fast_dev_run` mode: will run the requested loop using 1 batch(es). Logging and checkpointing is suppressed.


C:\Users\Chaco\OneDrive\Documents\GitHub\MVTecDestilado\.venv\Lib\site-packages\lightning\pytorch\callbacks\model_checkpoint.py:881: Checkpoint directory C:\Users\Chaco\OneDrive\Documents\GitHub\MVTecDestilado\outputs\checkpoints\B-cnn_distill-stfpm_l23_lr3e-4 exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


C:\Users\Chaco\OneDrive\Documents\GitHub\MVTecDestilado\.venv\Lib\site-packages\lightning\pytorch\utilities\model_summary\model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name    | Type          | Params | Mode  | FLOPs
----------------------------------------------------------
0 | net     | CNNClassifier | 718 K  | train | 0    
1 | teacher | ResNet18Stem  | 683 K  | eval  | 0    
----------------------------------------------------------
718 K     Trainable params
683 K     Non-trainable params
1.4 M     Total params
5.607     Total estimated model params size (MB)
44        Modules in train mode
36        Modules in eval mode
0         Total Flops


C:\Users\Chaco\OneDrive\Documents\GitHub\MVTecDestilado\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
C:\Users\Chaco\OneDrive\Documents\GitHub\MVTecDestilado\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
C:\Users\Chaco\OneDrive\Documents\GitHub\MVTecDestilado\.venv\Lib\site-packages\lightning\pytorch\loops\fit_loop.py:538: Found 36 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.


epoca   1 | train/acc 0.2500 | train/ce 2.2692 | train/distill 1.1228 | train/distill_conv2_x 0.4754 | train/distill_conv3_x 0.6474 | train/loss 3.3921 | val/acc 1.0000 | val/ce 2.1779 | val/distill 1.0881 | val/distill_conv2_x 0.4582 | val/distill_conv3_x 0.6299 | val/loss 3.2660


`Trainer.fit` stopped: `max_steps=1` reached.


fast_dev_run OK | teacher sigue en eval: True


### 3.7 Entrenamiento
Se entrena cada combinación con `run_experiment`, igual que el modelo A. Si una corrida ya se hizo, es decir si existen `outputs/results/<corrida>.json` y su checkpoint, se omite su entrenamiento y se carga el resumen guardado. Para repetir el entrenamiento se pone `RETRAIN = True`.

In [17]:
# Train model B with every combination (runs that already have a results file are loaded, not retrained)
RETRAIN = False
results_b = {}
for experiment in EXPERIMENTS_B:
    cfg_run = load_config([f"+experiment={experiment}"])
    results_b[cfg_run.name] = run_experiment(cfg_run, retrain=RETRAIN)

B-cnn_distill-stfpm_l23_lr3e-4: omitida, ya fue entrenada. Se cargan sus resultados de B-cnn_distill-stfpm_l23_lr3e-4.json y su checkpoint es outputs/checkpoints/B-cnn_distill-stfpm_l23_lr3e-4/best.ckpt
B-cnn_distill-stfpm_l3_lr3e-4: omitida, ya fue entrenada. Se cargan sus resultados de B-cnn_distill-stfpm_l3_lr3e-4.json y su checkpoint es outputs/checkpoints/B-cnn_distill-stfpm_l3_lr3e-4/best.ckpt
B-cnn_distill-stfpm_l23_sgd5e-2: omitida, ya fue entrenada. Se cargan sus resultados de B-cnn_distill-stfpm_l23_sgd5e-2.json y su checkpoint es outputs/checkpoints/B-cnn_distill-stfpm_l23_sgd5e-2/best.ckpt


### 3.8 Resultados y análisis de overfitting
La tabla resume cada corrida en su mejor época (la del checkpoint). La columna que permite comparar las tres combinaciones es `val/distill_conv3_x`, la pérdida STFPM en la etapa del embedding sobre imágenes de validación. El primer gráfico tiene una columna por corrida, con la pérdida total, la pérdida STFPM en `conv3_x` y el accuracy de entrenamiento y validación por época. El segundo compara `val/distill_conv3_x` de las tres corridas. Los gráficos se hacen con plotly. Quedan guardados dentro del notebook, así que se ven sin una cuenta de W&B, y son interactivos (zoom y valores al pasar el mouse). La misma figura se registra en una run de análisis de W&B, cuyo link queda impreso. Esa run tiene un id fijo, así que volver a ejecutar la celda la actualiza en lugar de crear otra. Las curvas de B se registran en la run `analisis-modelo-B`. Si la pérdida de validación sube mientras la de entrenamiento sigue bajando, el student está memorizando las imágenes de entrenamiento en lugar de aprender a imitar al teacher en general.

In [18]:
# Summary of the model B runs
with pd.option_context("display.float_format", "{:.4g}".format):
    display(runs_table(results_b, metrics=("loss", "distill_conv3_x", "ce", "acc")))

,epocas,parada temprana,mejor epoca,train/loss,val/loss,test/loss,train/distill_conv3_x,val/distill_conv3_x,test/distill_conv3_x,train/ce,val/ce,test/ce,train/acc,val/acc,test/acc,minutos
corrida,,,,,,,,,,,,,,,,
B-cnn_distill-stfpm_l23_lr3e-4,50,False,48,0.2492,0.24,0.2595,0.1567,0.1517,0.1653,0.002644,0.0007763,0.001146,1,1,1,3.36
B-cnn_distill-stfpm_l3_lr3e-4,50,False,48,0.1591,0.153,0.1677,0.1565,0.1523,0.1665,0.002534,0.0007541,0.00112,1,1,1,2.71
B-cnn_distill-stfpm_l23_sgd5e-2,50,False,48,0.3301,0.32,0.3413,0.2107,0.207,0.2219,0.006177,0.0007974,0.001091,1,1,1,3.55


In [19]:
# Training and validation curves of model B (plotly), also logged to W&B
fig_b_curves = train_val_figure(results_b, [("loss", "perdida total", False), ("distill_conv3_x", "STFPM en conv3_x", False),
                                            ("acc", "accuracy", False)],
                                title="Modelo B: entrenamiento y validacion por epoca")
fig_b_val = runs_figure(results_b, "val/distill_conv3_x", "STFPM en conv3_x (validacion)",
                        title="Modelo B: perdida STFPM de validacion en conv3_x de las tres corridas")
fig_b_curves.show()
fig_b_val.show()
log_figures(cfg, "analisis-modelo-b", "analisis-modelo-B", "Curvas de entrenamiento y validacion del modelo B",
            {"curvas/train_val": fig_b_curves, "curvas/val_distill_conv3_x": fig_b_val})

wandb: setting up run analisis-modelo-b


wandb: Tracking run with wandb version 0.31.2


wandb: Run data is saved locally in C:\Users\Chaco\OneDrive\Documents\GitHub\MVTecDestilado\outputs\wandb\run-20261127_121534-analisis-modelo-b
wandb: Run `wandb offline` to turn off syncing.


wandb: Resuming run analisis-modelo-B


wandb:  View project at https://forge.coreweave.com/wandb/danielpulido-instituto-tecnol-gico-de-costa-rica/MVTecDestilado


wandb:  View run at https://forge.coreweave.com/wandb/danielpulido-instituto-tecnol-gico-de-costa-rica/MVTecDestilado/runs/analisis-modelo-b


... (3 outputs left out)

wandb: uploading history steps 4-4, summary


wandb:  View run analisis-modelo-B at: https://forge.coreweave.com/wandb/danielpulido-instituto-tecnol-gico-de-costa-rica/MVTecDestilado/runs/analisis-modelo-b
wandb:  View project at: https://forge.coreweave.com/wandb/danielpulido-instituto-tecnol-gico-de-costa-rica/MVTecDestilado
wandb: Synced 4 W&B file(s), 2 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: C:\Users\Chaco\OneDrive\Documents\GitHub\MVTecDestilado\outputs\wandb\run-20261127_121534-analisis-modelo-b\logs


Graficos registrados en W&B: https://forge.coreweave.com/wandb/danielpulido-instituto-tecnol-gico-de-costa-rica/MVTecDestilado/runs/analisis-modelo-b


**Análisis.**

Las tres corridas convergen. Completan las 50 épocas sin activar `EarlyStopping`, su mejor época es la 48 y en las últimas épocas las curvas son planas, porque el coseno ya llevó el learning rate cerca de su mínimo. Por ejemplo, la `val/distill_conv3_x` de `stfpm_l23_lr3e-4` se mueve entre 0.1517 y 0.1527 desde la época 44. El student aprende a imitar al teacher. Su pérdida STFPM en `conv3_x` sobre validación baja de 0.61 en la época 1 a 0.1517 en la mejor época de la referencia, y como la pérdida equivale a $1 - \cos$, la similitud coseno promedio entre las features del student y las del teacher pasa de 0.39 a 0.85.

Supervisar también `conv2_x` casi no cambia el resultado en `conv3_x`. La mejor `val/distill_conv3_x` es 0.1517 con STFPM en las dos etapas y 0.1523 solo en `conv3_x`, una diferencia de 0.0006. La etapa intermedia sí cambia mucho, porque con supervisión su pérdida queda en 0.088 y sin ella en 0.556, por encima incluso del 0.46 del student aleatorio de la sección 3.3. El student llega a la misma representación final que el teacher por un camino intermedio distinto. El optimizador, en cambio, sí importa. Con SGD la mejor `val/distill_conv3_x` es 0.207 (similitud coseno de 0.79) frente a 0.1517 con AdamW, y además aprende más lento, ya que en la época 20 va en 0.258 y AdamW en 0.195.

Los ajustes hechos tras la primera ronda funcionaron. En la época 22 sigue apareciendo una perturbación, pero es mucho menor. La entropía cruzada de validación llega a 0.027 en `stfpm_l3_lr3e-4` y a 0.186 en `stfpm_l23_sgd5e-2`, frente a 0.51 y 1.41 con learning rate de 1e-3, y en `stfpm_l23_lr3e-4` no aparece. Las corridas se recuperan en una a tres épocas y la paciencia de 10 evita que se detengan antes de converger. Tampoco hay sobreajuste. En las tres corridas la pérdida STFPM de validación queda por debajo de la de entrenamiento (0.1517 frente a 0.1567 en la referencia), lo contrario de lo que haría un student que memorizara las imágenes de entrenamiento. La pérdida de prueba es algo mayor (0.1653), lo que es esperable porque 893 de las 1194 imágenes de prueba tienen defectos y en ellas el student no tiene por qué imitar igual de bien al teacher.

Sumar la pérdida STFPM no perjudica la clasificación. Las tres corridas terminan con accuracy de 1.000 en entrenamiento, validación y prueba, y con una entropía cruzada por debajo de 0.007, igual que el modelo A. La combinación elegida es `B-cnn_distill-stfpm_l23_lr3e-4`, porque tiene la menor `val/distill_conv3_x` (0.1517), y es la que representa al modelo B en las comparaciones siguientes. `stfpm_l3_lr3e-4` queda prácticamente empatada y entrena algo más rápido (2.7 minutos frente a 3.4). La diferencia entre la pérdida de prueba y la de validación es la que aprovecha la detección de anomalías, que se estudia en `anomalias.ipynb`.

## 4. Comparación de los modelos A y B

Esta sección compara el modelo A (entrenado desde cero), el modelo B (destilado con STFPM) y el teacher, el ResNet-18 completo preentrenado en ImageNet. Se miden tres cosas:

- **Costo:** parámetros, tamaño en memoria y operaciones por imagen.
- **Velocidad:** tiempo de inferencia para obtener el embedding.
- **Qué aprendió cada student:** qué tan parecidas son sus features a las del teacher, y cómo evolucionó el entrenamiento de A y B.

La capacidad de cada modelo para detectar anomalías no se mide aquí. Se mide en `anomalias.ipynb` con los embeddings de estos mismos checkpoints. Los gráficos se muestran en cada subsección y al final se registran todos en la run de análisis `analisis-comparacion-AB` de W&B.

### 4.1 Modelos que se comparan
De cada modelo se usa la combinación elegida en su sección:

- **Modelo A:** `A-cnn_scratch-lr1e-3cos_bs64`, la de menor pérdida de validación (4.5e-5). Como la clasificación se satura, las tres corridas de A son prácticamente equivalentes.
- **Modelo B:** `B-cnn_distill-stfpm_l23_lr3e-4`, la de menor pérdida STFPM de validación en `conv3_x` (0.1517).
- **Teacher completo:** el ResNet-18 de ImageNet hasta su *average pooling*, con un embedding de 512 dimensiones.
- **Teacher hasta `conv3_x`:** las mismas etapas que tienen A y B, con los pesos de ImageNet. Es la parte del teacher que B imita.

Los checkpoints se cargan con `load_trained_model`, que reconstruye el modelo desde su configuración de Hydra y carga los pesos del mejor checkpoint.

In [20]:
# Load the chosen checkpoints of models A and B and build the teacher references
from torchvision.models import resnet18

from src.comparison import count_flops, inference_time_ms, teacher_similarity
from src.training import load_trained_model

BEST_A, BEST_B = "cnn_scratch/lr1e-3cos_bs64", "cnn_distill/stfpm_l23"
model_a_best = load_trained_model(load_config([f"+experiment={BEST_A}"]))
model_b_best = load_trained_model(load_config([f"+experiment={BEST_B}"]))

teacher_full = resnet18(weights=cfg_b.model.teacher.weights).eval()
teacher_embedding = torch.nn.Sequential(*list(teacher_full.children())[:-1], torch.nn.Flatten())  # up to avgpool
teacher_stem = model_b_best.teacher  # conv1 to conv3_x with the ImageNet weights

MODELS = {
    "teacher completo": teacher_embedding,
    "teacher hasta conv3_x": torch.nn.Sequential(teacher_stem, torch.nn.AdaptiveAvgPool2d(1), torch.nn.Flatten()),
    "modelo A": model_a_best.net,
    "modelo B": model_b_best.net,
}
EMBEDDING = {
    "teacher completo": teacher_embedding,
    "teacher hasta conv3_x": MODELS["teacher hasta conv3_x"],
    "modelo A": model_a_best.get_embedding,
    "modelo B": model_b_best.get_embedding,
}
with torch.no_grad():
    for name, fn in EMBEDDING.items():
        print(f"{name:22s} embedding {tuple(fn(batch['image'][:2]).shape)}")

FIGURES_AB = {}  # charts of this section, logged to W&B at the end

teacher completo       embedding (2, 512)
teacher hasta conv3_x  embedding (2, 128)
modelo A               embedding (2, 128)
modelo B               embedding (2, 128)


### 4.2 Costo: parámetros, memoria y operaciones
Para cada modelo se cuentan sus parámetros, su tamaño en memoria con pesos de 32 bits (4 bytes por parámetro) y los GFLOPs necesarios para calcular el embedding de una imagen de 224×224. Los FLOPs se cuentan con `torch.utils.flop_counter.FlopCounterMode`, que suma las operaciones de cada capa (una multiplicación y una suma cuentan como 2). Los parámetros de A y B incluyen su clasificador FC, que no se usa para el embedding pero sí es parte del modelo entrenado.

In [21]:
# Parameters, memory and GFLOPs per image of each model
cost = pd.DataFrame({
    "modelo": list(MODELS),
    "parametros (M)": [count_parameters(m) / 1e6 for m in MODELS.values()],
    "GFLOPs por imagen": [count_flops(fn) / 1e9 for fn in EMBEDDING.values()],
    "dimension del embedding": [EMBEDDING[name](batch["image"][:1]).shape[1] for name in MODELS],
})
cost["memoria (MB)"] = cost["parametros (M)"] * 4
reference = cost.set_index("modelo").loc["teacher completo"]
cost["% parametros del teacher"] = 100 * cost["parametros (M)"] / reference["parametros (M)"]
cost["% GFLOPs del teacher"] = 100 * cost["GFLOPs por imagen"] / reference["GFLOPs por imagen"]

FIGURES_AB["costo/parametros"] = bar_figure(cost, "modelo", "parametros (M)", "Parametros (millones)")
FIGURES_AB["costo/gflops"] = bar_figure(cost, "modelo", "GFLOPs por imagen", "GFLOPs por imagen de 224x224")
display(cost.set_index("modelo").round(3))
FIGURES_AB["costo/parametros"].show()
FIGURES_AB["costo/gflops"].show()

,parametros (M),GFLOPs por imagen,dimension del embedding,memoria (MB),% parametros del teacher,% GFLOPs del teacher
modelo,,,,,,
teacher completo,11.177,3.627,512,44.706,100.000,100.00
teacher hasta conv3_x,0.683,1.983,128,2.732,6.112,54.67
modelo A,0.719,1.983,128,2.875,6.430,54.67
modelo B,0.719,1.983,128,2.875,6.430,54.67


### 4.3 Velocidad: tiempo de inferencia
Se mide el tiempo que tarda cada modelo en calcular el embedding en dos escenarios:

- **GPU con batch de 32:** el caso de procesar lotes de imágenes, por ejemplo una inspección por tandas.
- **CPU con una imagen:** el caso de un equipo sin GPU que inspecciona pieza por pieza.

Se descartan 10 repeticiones de calentamiento y se toma la mediana de las siguientes (50 en GPU y 20 en CPU), que es menos sensible que el promedio a repeticiones lentas aisladas. En GPU se llama `torch.cuda.synchronize()` antes de leer el reloj, porque las operaciones de CUDA son asíncronas. Los tiempos dependen del equipo (RTX 4060 y el CPU de esta máquina), así que lo comparable es la proporción entre modelos y no el valor absoluto.

In [22]:
# Median inference time of the embedding on GPU (batch 32) and CPU (one image)
times = []
for name, model in MODELS.items():
    fn = EMBEDDING[name]
    model.to("cuda")
    gpu_ms = inference_time_ms(fn, batch_size=32, device="cuda")
    model.to("cpu")
    cpu_ms = inference_time_ms(fn, batch_size=1, device="cpu", repeats=20)
    times.append({"modelo": name, "GPU batch 32 (ms)": gpu_ms, "GPU imagenes por segundo": 32 * 1000 / gpu_ms,
                  "CPU 1 imagen (ms)": cpu_ms})
times = pd.DataFrame(times)
teacher_times = times.set_index("modelo").loc["teacher completo"]
times["% tiempo GPU del teacher"] = 100 * times["GPU batch 32 (ms)"] / teacher_times["GPU batch 32 (ms)"]
times["% tiempo CPU del teacher"] = 100 * times["CPU 1 imagen (ms)"] / teacher_times["CPU 1 imagen (ms)"]

FIGURES_AB["velocidad/gpu_ms"] = bar_figure(times, "modelo", "GPU batch 32 (ms)",
                                            "Tiempo de inferencia en GPU, batch de 32 (ms)", ".1f")
FIGURES_AB["velocidad/cpu_ms"] = bar_figure(times, "modelo", "CPU 1 imagen (ms)",
                                            "Tiempo de inferencia en CPU, una imagen (ms)", ".1f")
display(times.set_index("modelo").round(2))
FIGURES_AB["velocidad/gpu_ms"].show()
FIGURES_AB["velocidad/cpu_ms"].show()

,GPU batch 32 (ms),GPU imagenes por segundo,CPU 1 imagen (ms),% tiempo GPU del teacher,% tiempo CPU del teacher
modelo,,,,,
teacher completo,16.38,1953.73,18.77,100.00,100.00
teacher hasta conv3_x,11.49,2784.52,9.97,70.16,53.10
modelo A,11.51,2780.55,9.52,70.26,50.71
modelo B,11.49,2784.81,10.28,70.16,54.76


### 4.4 Similitud de las features con el teacher
Esta medición muestra qué cambió el destilado en el student. Para cada corrida de A y de B se calcula la pérdida STFPM entre sus features y las del teacher en `conv2_x` y `conv3_x`, promediada sobre todas las imágenes de validación. Como la pérdida es $1 - \cos$, un valor de 0 significa features iguales en dirección y 1 significa features ortogonales. También se incluye la misma red con pesos aleatorios, sin entrenar, como referencia.

El modelo A nunca vio al teacher, así que su valor indica qué tan parecido a ImageNet es lo que aprendió por su cuenta al clasificar las categorías.

In [23]:
# STFPM loss (1 - cosine) between the features of every run and the teacher, on the validation images
from src.models.cnn import ResNet18Stem

torch.manual_seed(cfg.seed)
backbones = {"sin entrenar (pesos aleatorios)": ResNet18Stem()}
for experiment in EXPERIMENTS_A + EXPERIMENTS_B:
    c = load_config([f"+experiment={experiment}"])
    backbones[c.name] = load_trained_model(c).net.backbone

similarity = pd.DataFrame([
    {"backbone": name, **{f"STFPM {k}": v for k, v in teacher_similarity(bb, teacher_stem, datamodule.val_dataloader()).items()}}
    for name, bb in backbones.items()
])
similarity["coseno medio conv3_x"] = 1 - similarity["STFPM conv3_x"]
teacher_stem.to("cpu")

FIGURES_AB["similitud/coseno_conv3_x"] = bar_figure(similarity, "backbone", "coseno medio conv3_x",
                                                    "Similitud coseno media con el teacher en conv3_x (validacion)")
FIGURES_AB["similitud/stfpm_conv2_x"] = bar_figure(similarity, "backbone", "STFPM conv2_x",
                                                   "Perdida STFPM con el teacher en conv2_x (validacion)")
display(similarity.set_index("backbone").round(4))
FIGURES_AB["similitud/coseno_conv3_x"].show()
FIGURES_AB["similitud/stfpm_conv2_x"].show()

,STFPM conv2_x,STFPM conv3_x,coseno medio conv3_x
backbone,,,
sin entrenar (pesos aleatorios),0.5304,0.6295,0.3705
A-cnn_scratch-lr1e-3_bs32,0.5165,0.7230,0.2770
A-cnn_scratch-lr3e-4_bs32,0.5093,0.7122,0.2878
A-cnn_scratch-lr1e-3cos_bs64,0.5317,0.7060,0.2940
B-cnn_distill-stfpm_l23_lr3e-4,0.0875,0.1517,0.8483
B-cnn_distill-stfpm_l3_lr3e-4,0.5553,0.1523,0.8477
B-cnn_distill-stfpm_l23_sgd5e-2,0.1122,0.2069,0.7931


### 4.5 Entrenamiento de A y B
Las pérdidas totales de A y B no se pueden comparar directamente, porque la de B suma la pérdida STFPM. Lo que sí comparten es la entropía cruzada de la clasificación y el accuracy. Los gráficos muestran, para la corrida elegida de cada modelo, la entropía cruzada (en escala logarítmica) y el accuracy de entrenamiento y validación por época, y la entropía cruzada de validación de los dos modelos juntos. Al final de la celda se registran en W&B todos los gráficos de la sección.

In [24]:
# Cross entropy and accuracy curves of the chosen runs of A and B, and the W&B charts of the section
name_a = load_config([f"+experiment={BEST_A}"]).name
name_b = load_config([f"+experiment={BEST_B}"]).name
# In model A the total loss is the cross entropy, so it is renamed to compare it with the "ce" of model B
history_a = [{**h, "train/ce": h["train/loss"], "val/ce": h["val/loss"]} for h in results_a[name_a]["history"]]
results_ab = {name_a: {**results_a[name_a], "history": history_a}, name_b: results_b[name_b]}

FIGURES_AB["curvas/train_val"] = train_val_figure(
    results_ab, [("ce", "entropia cruzada", True), ("acc", "accuracy", False)],
    title="Modelos A y B: entropia cruzada y accuracy por epoca")
FIGURES_AB["curvas/val_ce"] = runs_figure(results_ab, "val/ce", "entropia cruzada de validacion", log_y=True,
                                          title="Entropia cruzada de validacion de A y B")
FIGURES_AB["curvas/train_val"].show()
FIGURES_AB["curvas/val_ce"].show()
log_figures(cfg, "analisis-comparacion-ab", "analisis-comparacion-AB",
            "Comparacion de costo, velocidad y similitud con el teacher de los modelos A y B", FIGURES_AB)

wandb: setting up run analisis-comparacion-ab


wandb: Tracking run with wandb version 0.31.2


wandb: Run data is saved locally in C:\Users\Chaco\OneDrive\Documents\GitHub\MVTecDestilado\outputs\wandb\run-20261127_121552-analisis-comparacion-ab
wandb: Run `wandb offline` to turn off syncing.


wandb: Resuming run analisis-comparacion-AB


wandb:  View project at https://forge.coreweave.com/wandb/danielpulido-instituto-tecnol-gico-de-costa-rica/MVTecDestilado


wandb:  View run at https://forge.coreweave.com/wandb/danielpulido-instituto-tecnol-gico-de-costa-rica/MVTecDestilado/runs/analisis-comparacion-ab


wandb: uploading summary; updating run metadata


wandb: uploading config.yaml; uploading wandb-metadata.json; uploading requirements.txt; uploading media\plotly\costo\gflops_16_ec41eef64e4e9dd19738.plotly.json; uploading media\plotly\velocidad\gpu_ms_16_45c69f53b73344db2d47.plotly.json (+ 7 more)


wandb:  View run analisis-comparacion-AB at: https://forge.coreweave.com/wandb/danielpulido-instituto-tecnol-gico-de-costa-rica/MVTecDestilado/runs/analisis-comparacion-ab
wandb:  View project at: https://forge.coreweave.com/wandb/danielpulido-instituto-tecnol-gico-de-costa-rica/MVTecDestilado
wandb: Synced 4 W&B file(s), 8 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: C:\Users\Chaco\OneDrive\Documents\GitHub\MVTecDestilado\outputs\wandb\run-20261127_121552-analisis-comparacion-ab\logs


Graficos registrados en W&B: https://forge.coreweave.com/wandb/danielpulido-instituto-tecnol-gico-de-costa-rica/MVTecDestilado/runs/analisis-comparacion-ab


**Análisis.**

Los modelos A y B tienen exactamente el mismo costo porque comparten arquitectura. Los dos tienen 0.719 millones de parámetros (2.9 MB con pesos de 32 bits), un embedding de 128 dimensiones y 1.98 GFLOPs por imagen, ya que el teacher solo se usa durante el entrenamiento y no forma parte del modelo final. El teacher completo tiene 11.18 millones de parámetros hasta el *average pooling* (11.69 millones con su clasificador, como en la sección 2.2) y 3.63 GFLOPs. La reducción de parámetros es mucho mayor que la de cómputo. A y B tienen el 6.4 % de los parámetros del teacher, pero necesitan el 54.7 % de sus operaciones, porque las primeras etapas de una ResNet tienen pocos filtros pero trabajan sobre mapas grandes (112×112 y 56×56), mientras que las últimas concentran la mayoría de los parámetros sobre mapas de 14×14 y 7×7.

La velocidad sigue al cómputo y no a los parámetros. En GPU, con batch de 32, A y B tardan 11.5 ms frente a 16.4 ms del teacher, es decir el 70 % del tiempo, con unas 2780 imágenes por segundo frente a 1950. En CPU, con una imagen, tardan entre 9.5 y 10.3 ms frente a 18.8 ms, entre el 51 % y el 55 % del tiempo. En CPU la ganancia se acerca más a la reducción de FLOPs porque una sola imagen no alcanza a ocupar la GPU, donde pesan más los costos fijos de lanzar cada operación. Estos tiempos varían unas décimas de milisegundo entre ejecuciones del notebook, pero las proporciones se mantienen.

La diferencia entre A y B está en lo que aprende cada uno. La similitud coseno promedio entre sus features de `conv3_x` y las del teacher es de 0.85 en `B-cnn_distill-stfpm_l23_lr3e-4` y de solo 0.28 a 0.29 en las tres corridas de A, por debajo incluso del 0.37 de la red sin entrenar. Al clasificar por su cuenta, el modelo A se aleja de lo que hace el teacher, mientras que el modelo B, con la misma arquitectura y los mismos datos, llega a features muy parecidas a las de ImageNet. Las tres corridas de A se parecen entre sí, con pérdidas STFPM de 0.706 a 0.723 en `conv3_x` y de 0.509 a 0.532 en `conv2_x`, valores cercanos a los de la red sin entrenar (0.630 y 0.530). Sus hiperparámetros no cambian esa relación, lo que coincide con lo observado en la sección 2.7.

B clasifica igual que A y entrena de forma más estable. Los dos llegan a un accuracy de 1.000 en validación. La entropía cruzada de validación de A baja más (4.5e-5 frente a 7.8e-4 en B, en sus mejores épocas) porque B reparte el entrenamiento entre clasificar e imitar al teacher, pero A tiene picos de hasta 0.74 en las primeras épocas, mientras que la de B no pasa de 0.011 después de la época 3. En conjunto, con el mismo costo que A, el modelo B representa las imágenes como lo hace el ResNet-18 en sus primeras etapas, con el 6.4 % de sus parámetros y entre el 51 % y el 70 % de su tiempo de inferencia. Falta comprobar en `anomalias.ipynb` si esas features más parecidas a las del teacher producen un mejor detector de anomalías, que es lo que la hipótesis del proyecto necesita.